# Traffic vehicle detection

Seven classes annotated in the Kaggle traffic-vehicles dataset: Car, Number Plate,
Blur Number Plate, Two Wheeler, Auto, Bus, Truck.

The notebook trains a YOLO11 detector on them and exports it to ONNX for the browser
front-end. It assumes a T4 runtime; on CPU the training step runs for hours instead of
half an hour.

In [ ]:
%pip install -q ultralytics onnx onnxslim onnxruntime
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Dataset

The archive ships its own split: `images/{train,val,test}`, with labels for train and val
only. 732 training images carry 9 153 boxes, 184 validation images carry 1 980. The test
folder holds 267 unlabelled images and 18 videos, so it plays no part in training.

Class balance is uneven. Car alone accounts for 47 % of the boxes, Bus and Truck for
roughly 4 % each.

The cell below prompts for `archive.zip` unless the file is already in the session.

In [ ]:
import zipfile, yaml
from pathlib import Path

if not Path('archive.zip').exists():
    from google.colab import files
    files.upload()

ROOT = Path('/content/data')
if not ROOT.exists():
    with zipfile.ZipFile('archive.zip') as z:
        z.extractall(ROOT)

BASE = next(p.parent.parent for p in ROOT.rglob('images/train') if p.is_dir())
CLASS_NAMES = ['Car', 'Number Plate', 'Blur Number Plate', 'Two Wheeler', 'Auto', 'Bus', 'Truck']

DATA_YAML = Path('/content/data.yaml')
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(BASE), 'train': 'images/train', 'val': 'images/val',
    'nc': len(CLASS_NAMES), 'names': dict(enumerate(CLASS_NAMES)),
}, sort_keys=False))

for split in ('train', 'val'):
    print(split, len(list((BASE / 'labels' / split).glob('*.txt'))), 'labelled images')

## Training

`yolo11s`, 100 epochs at 640 px, batch 16. Roughly 35 minutes on a T4, around 22 seconds
per epoch once the dataset cache is warm.

Three of the seven classes exist in COCO, so their weights carry over from the pretrained
head. The other four start from scratch.

Augmentations are set for road scenes: horizontal flips only, rotation capped at 5
degrees, mosaic disabled over the last ten epochs. `patience=25` cuts the run short if the
mAP stops moving, and the checkpoint kept is the best epoch, not the last one.

Model size drives the download the front-end has to pull: `yolo11n` exports to about
10 MB, `yolo11s` to 38 MB, `yolo11m` to 80 MB.

In [ ]:
from ultralytics import YOLO

IMGSZ, EPOCHS = 640, 100

results = YOLO('yolo11s.pt').train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=16,
    device=0, workers=2, patience=25, seed=42,
    project='/content/runs', name='traffic-yolo', exist_ok=True, plots=True,
    # road scenes: horizontal flips only, mild rotation
    fliplr=0.5, flipud=0.0, degrees=5.0, translate=0.1, scale=0.5, shear=2.0,
    mosaic=1.0, close_mosaic=10,
)
BEST = Path(results.save_dir) / 'weights' / 'best.pt'

## Results

`class_result(i)` indexes the classes that actually appear in the split, in the order of
`ap_class_index`. Passing the absolute class id instead shifts every metric that follows a
class with no instances, without raising anything.

In [ ]:
from IPython.display import Image, display

for name in ('results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
    p = Path(results.save_dir) / name
    if p.exists():
        print(name); display(Image(filename=str(p), width=900))

m = YOLO(BEST).val(data=str(DATA_YAML), device=0, split='val')

# class_result(i) indexes the evaluated classes, not the absolute class id
order = {int(c): i for i, c in enumerate(m.box.ap_class_index)}
print(f"\n{'class':<20}{'P':>8}{'R':>8}{'mAP50':>10}{'mAP50-95':>10}")
for i, name in enumerate(CLASS_NAMES):
    if i in order:
        p, r, ap50, ap = m.box.class_result(order[i])
        print(f'{name:<20}{p:>8.3f}{r:>8.3f}{ap50:>10.3f}{ap:>10.3f}')
print(f'\nmAP50 {m.box.map50:.4f}   mAP50-95 {m.box.map:.4f}')

## ONNX export

Exported with `nms=False` and `dynamic=False`. onnxruntime-web covers the NMS operators
only partially, so the browser decodes the raw output and runs non-maximum suppression
itself. Opset 12 is what the WASM backend handles reliably.

The output shape is `[1, 11, 8400]`: four box coordinates plus seven class scores, over
80² + 40² + 20² candidate positions.

In [ ]:
import onnxruntime as ort

onnx_path = YOLO(BEST).export(format='onnx', imgsz=IMGSZ, opset=12,
                              simplify=True, dynamic=False, nms=False, half=False)

s = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
print('input', s.get_inputs()[0].shape, '-> output', s.get_outputs()[0].shape)
print(f'{Path(onnx_path).stat().st_size / 1e6:.1f} MB')

## Weights

`best.onnx` is what the front-end loads. `best.pt` remains the PyTorch source for any
later re-export at a different resolution.

In [ ]:
from google.colab import files

files.download(str(onnx_path))
files.download(str(BEST))